In [57]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from contextlib import AsyncExitStack
import json
from jsonschema import validate,ValidationError
from typing import Annotated, Sequence, TypedDict, Any, Dict,Union
from langchain_core.messages import BaseMessage,SystemMessage,AIMessage,ToolMessage,HumanMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from langchain_core.tools import StructuredTool
from pydantic import create_model
import nest_asyncio 
from dotenv import load_dotenv
import os


In [58]:
load_dotenv()

True

In [59]:
def remove_descriptions(data, max_length=None):
    """
    Recursively remove description fields from JSON schema.
    If max_length is set, remove only descriptions longer than max_length.
    """
    if isinstance(data, dict):
        new_dict = {}
        for key, value in data.items():


            if key == "description":
                if max_length is None:  
                    continue
                elif isinstance(value, str) and len(value) > max_length:
                    continue  

            new_dict[key] = remove_descriptions(value, max_length)
        return new_dict

    elif isinstance(data, list):
        return [remove_descriptions(item, max_length) for item in data]

    return data


def validate_arguments(inputs_args, schema):
    try:
        validate(instance=inputs_args, schema=schema)
        print(f"---------Valid Arguments---------")
        return "Valid"
    except ValidationError as e:
        print(f"---------Invalid Arguments-------")
        return f"Invalid as {e}"

In [60]:
from typing import List,Union
MAP = {
    "string":str,
    "number":float,
    "integer":int,
    "boolean":bool,
    "array":List[str],
    "object":dict
}


def json_to_model(name,schema):
    fields = {}
    required = schema.get("required",[])
    properties = schema.get("properties",{})


    for prop,rules in properties.items():
        if "anyOf" in rules:
            possible = []
            for option in rules["anyOf"]:
                if option["type"] == "string":
                    possible.append(MAP["string"])
                elif option["type"] == "array":
                    possible.append(MAP["array"])

            fields[prop] = (Union[tuple(possible)],... if prop in required else None)

        else:
            py_type = MAP[rules["type"]]
            fields[prop] = (py_type, ... if prop in required else None)

    return create_model(name, **fields)


In [61]:
async def mcp_execute(session, tool_name: str, **kwargs):
    """Generic executor for ANY MCP tool."""
    result = await session.call_tool(tool_name, kwargs)
    return result

In [62]:


def build_tool_from_schema(tool_name,tool_description,tool_schema,session):
    new_tool_name = tool_name.replace("-","_")
    new_tool_name = new_tool_name+"_Args"
    Arg_model = json_to_model(new_tool_name,tool_schema)

    async def wrapper(**kwargs):
        try:
            return await mcp_execute(
                session=session,
                tool_name=tool_name,
                **kwargs
            )
        except Exception as e:
            return f"TOOL ERROR: {type(e).__name__}: {str(e)}"


    nest_asyncio.apply()

    def sync_wrapper(**kwargs):
        import asyncio
        return asyncio.get_event_loop().run_until_complete(wrapper(**kwargs))

    tool = StructuredTool.from_function(
        name=tool_name,
        description=tool_description,
        func=sync_wrapper,
        args_schema=Arg_model
    )

    return tool

In [63]:
def load_config() -> Union[Dict, None]:
    config_path = "mcp.json"

    try:
        with open(config_path) as f:
            config = json.load(f)

            mcp_servers = config.get("mcpServers", {})

            if not mcp_servers:
                print("No MCP servers found")

            return mcp_servers

    except Exception as e:
        print(f"Unable to open Config at path {config_path} as {e}")
        return None

async def configure_mcp(mcp_servers):


    input_schemas = {}
    name_to_tool = {}
    tools_list = []
    stack = AsyncExitStack()
    await stack.__aenter__()

    try:
        for server_name, server_info in mcp_servers.items():
            print(f"Connecting to server {server_name}...")

            server_param = StdioServerParameters(
                command=server_info["command"],
                args=server_info["args"],
                env=server_info.get("env")
            )

            read, write = await stack.enter_async_context(stdio_client(server_param))

            session = await stack.enter_async_context(
                ClientSession(read_stream=read, write_stream=write)
            )

            await session.initialize()
            print(f"Session initialized for {server_name}")



            server_tools = await session.list_tools()
            for tool in server_tools.tools:
                clean_schema = remove_descriptions(tool.inputSchema, max_length=200)  # or None
                input_schemas[tool.name] = clean_schema
                create_tool = build_tool_from_schema(tool.name,tool.description,clean_schema,session)
                name_to_tool[tool.name] = create_tool
                tools_list.append(create_tool)

        schema_path = "Input_schema.json"
        os.makedirs("schemas", exist_ok=True)
        with open("Input_schema.json", "w") as f:
            json.dump(input_schemas, f, indent=3)
            print(f"Schema Dumped at {schema_path}")

        return stack,input_schemas,tools_list,name_to_tool

    except Exception as e:
        print(f"Stack cloased due to some problem as {e}")
        
        await stack.aclose()

In [64]:
import json
import uuid
from typing import Dict, List, Optional, Any, Tuple
from datetime import datetime
from enum import Enum


class NodeStatus(Enum):
    """Enumeration of possible node statuses."""
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"
    BLOCKED = "blocked"
    VULNERABLE = "vulnerable"
    NOT_VULNERABLE = "not_vulnerable"


class RiskLevel(Enum):
    """Enumeration of risk levels."""
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"


class TaskNode:
    """Represents a single node in the task tree."""
    
    def __init__(
        self,
        description: str,
        parent_id: Optional[str] = None,
        node_type: str = "task",
        **kwargs
    ):
        """Initialize a task node."""
        self.id = kwargs.get('id', str(uuid.uuid4()))
        self.description = description
        self.status = NodeStatus(kwargs.get('status', NodeStatus.PENDING.value))
        self.node_type = node_type  # task, phase, finding, objective
        self.parent_id = parent_id
        self.children_ids: List[str] = kwargs.get('children_ids', [])
        
        # Task execution details
        self.tool_used = kwargs.get('tool_used', None)
        self.command_executed = kwargs.get('command_executed', None)
        self.output_summary = kwargs.get('output_summary', None)
        self.findings = kwargs.get('findings', None)
        
        # Metadata
        self.priority = kwargs.get('priority', 5)  # 1-10, higher is more important
        self.risk_level = RiskLevel(kwargs.get('risk_level', RiskLevel.LOW.value))
        self.timestamp = kwargs.get('timestamp', None)
        self.kb_references = kwargs.get('kb_references', [])
        self.dependencies = kwargs.get('dependencies', [])
        
        # Additional attributes
        self.attributes = kwargs.get('attributes', {})
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert node to dictionary representation."""
        return {
            'id': self.id,
            'description': self.description,
            'status': self.status.value,
            'node_type': self.node_type,
            'parent_id': self.parent_id,
            'children_ids': self.children_ids,
            'tool_used': self.tool_used,
            'command_executed': self.command_executed,
            'output_summary': self.output_summary,
            'findings': self.findings,
            'priority': self.priority,
            'risk_level': self.risk_level.value,
            'timestamp': self.timestamp,
            'kb_references': self.kb_references,
            'dependencies': self.dependencies,
            'attributes': self.attributes
        }
    
    @classmethod
    def from_dict(cls, data: Dict[str, Any]) -> 'TaskNode':
        """Create node from dictionary representation."""
        return cls(
            description=data['description'],
            **data
        )


class TaskTreeManager:
    """Manages the Pentesting Task Tree (PTT) structure and operations."""
    
    def __init__(self):
        """Initialize the task tree manager."""
        self.nodes: Dict[str, TaskNode] = {}
        self.root_id: Optional[str] = None
        self.goal: Optional[str] = None
        self.target: Optional[str] = None
        self.constraints: Dict[str, Any] = {}
        self.creation_time = datetime.now()
        
    def initialize_tree(self, goal: str, target: str, constraints: Dict[str, Any] = None) -> str:
        """
        Initialize the task tree with a goal and target.
        
        Args:
            goal: The primary objective
            target: The target system/network
            constraints: Any constraints or scope limitations
            
        Returns:
            The root node ID
        """
        self.goal = goal
        self.target = target
        self.constraints = constraints or {}
        
        # Create root node - let the LLM determine what structure is needed
        root_node = TaskNode(
            description=f"Goal: {goal}",
            node_type="objective"
        )
        self.root_id = root_node.id
        self.nodes[root_node.id] = root_node
        
        return self.root_id
    
    def add_node(self, node: TaskNode) -> str:
        """
        Add a node to the tree.
        
        Args:
            node: The TaskNode to add
            
        Returns:
            The node ID
        """
        self.nodes[node.id] = node
        
        # Update parent's children list
        if node.parent_id and node.parent_id in self.nodes:
            parent = self.nodes[node.parent_id]
            if node.id not in parent.children_ids:
                parent.children_ids.append(node.id)
        
        return node.id
    
    def update_node(self, node_id: str, updates: Dict[str, Any]) -> bool:
        """
        Update a node's attributes.
        
        Args:
            node_id: The ID of the node to update
            updates: Dictionary of attributes to update
            
        Returns:
            True if successful, False otherwise
        """
        if node_id not in self.nodes:
            return False
        
        node = self.nodes[node_id]
        
        # Update allowed fields
        allowed_fields = {
            'status', 'tool_used', 'command_executed', 'output_summary',
            'findings', 'priority', 'risk_level', 'timestamp', 'kb_references'
        }
        
        for field, value in updates.items():
            if field in allowed_fields:
                if field == 'status':
                    node.status = NodeStatus(value)
                elif field == 'risk_level':
                    node.risk_level = RiskLevel(value)
                else:
                    setattr(node, field, value)
            elif field == 'attributes':
                node.attributes.update(value)
        
        return True
    
    def get_node(self, node_id: str) -> Optional[TaskNode]:
        """Get a node by ID."""
        return self.nodes.get(node_id)
    
    def get_children(self, node_id: str) -> List[TaskNode]:
        """Get all children of a node."""
        if node_id not in self.nodes:
            return []
        
        parent = self.nodes[node_id]
        return [self.nodes[child_id] for child_id in parent.children_ids if child_id in self.nodes]
    
    def get_leaf_nodes(self) -> List[TaskNode]:
        """Get all leaf nodes (nodes without children)."""
        return [node for node in self.nodes.values() if not node.children_ids]
    
    def get_candidate_tasks(self) -> List[TaskNode]:
        """
        Get candidate tasks for next action.
        
        Returns tasks that are:
        - Leaf nodes
        - Status is PENDING or FAILED
        - All dependencies are completed
        """
        candidates = []
        
        for node in self.get_leaf_nodes():
            if node.status in [NodeStatus.PENDING, NodeStatus.FAILED]:
                # Check dependencies
                deps_satisfied = all(
                    self.nodes.get(dep_id, TaskNode("")).status == NodeStatus.COMPLETED
                    for dep_id in node.dependencies
                )
                
                if deps_satisfied:
                    candidates.append(node)
        
        return candidates
    
    def prioritize_tasks(self, tasks: List[TaskNode]) -> List[TaskNode]:
        """
        Prioritize tasks based on various factors.
        
        Args:
            tasks: List of candidate tasks
            
        Returns:
            Sorted list of tasks (highest priority first)
        """
        def task_score(task: TaskNode) -> float:
            # Base score from priority
            score = task.priority
            
            # Boost for reconnaissance tasks in early stages
            if "recon" in task.description.lower() or "scan" in task.description.lower():
                completed_count = sum(1 for n in self.nodes.values() if n.status == NodeStatus.COMPLETED)
                if completed_count < 5:
                    score += 3
            
            # Boost for vulnerability assessment after recon
            if "vuln" in task.description.lower() and self._has_completed_recon():
                score += 2
            
            # Penalty for high-risk tasks early on
            if task.risk_level == RiskLevel.HIGH:
                score -= 2
            
            return score
        
        return sorted(tasks, key=task_score, reverse=True)
    
    def _has_completed_recon(self) -> bool:
        """Check if basic reconnaissance has been completed."""
        recon_keywords = ["scan", "recon", "enumerat", "discover"]
        completed_recon = any(
            any(keyword in node.description.lower() for keyword in recon_keywords)
            and node.status == NodeStatus.COMPLETED
            for node in self.nodes.values()
        )
        return completed_recon
    
    def to_natural_language(self, node_id: Optional[str] = None, indent: int = 0) -> str:
        """
        Convert the tree (or subtree) to natural language representation.
        
        Args:
            node_id: Starting node ID (None for root)
            indent: Indentation level
            
        Returns:
            Natural language representation of the tree
        """
        if node_id is None:
            node_id = self.root_id
        
        if node_id not in self.nodes:
            return ""
        
        node = self.nodes[node_id]
        indent_str = "  " * indent
        
        # Format node information
        status_symbol = {
            NodeStatus.PENDING: "○",
            NodeStatus.IN_PROGRESS: "◐",
            NodeStatus.COMPLETED: "●",
            NodeStatus.FAILED: "✗",
            NodeStatus.BLOCKED: "□",
            NodeStatus.VULNERABLE: "⚠",
            NodeStatus.NOT_VULNERABLE: "✓"
        }.get(node.status, "?")
        
        lines = [f"{indent_str}{status_symbol} {node.description}"]
        
        # Add findings if present
        if node.findings:
            lines.append(f"{indent_str}  → Findings: {node.findings}")
        
        # Add tool/command info if present
        if node.tool_used:
            lines.append(f"{indent_str}  → Tool: {node.tool_used}")
        
        # Process children
        for child_id in node.children_ids:
            lines.append(self.to_natural_language(child_id, indent + 1))
        
        return "\n".join(lines)
    
    def to_json(self) -> str:
        """Serialize the tree to JSON."""
        data = {
            'goal': self.goal,
            'target': self.target,
            'constraints': self.constraints,
            'root_id': self.root_id,
            'creation_time': self.creation_time.isoformat(),
            'nodes': {node_id: node.to_dict() for node_id, node in self.nodes.items()}
        }
        return json.dumps(data, indent=2)
    
    @classmethod
    def from_json(cls, json_str: str) -> 'TaskTreeManager':
        """Deserialize a tree from JSON."""
        data = json.loads(json_str)
        
        manager = cls()
        manager.goal = data['goal']
        manager.target = data['target']
        manager.constraints = data['constraints']
        manager.root_id = data['root_id']
        manager.creation_time = datetime.fromisoformat(data['creation_time'])
        
        # Recreate nodes
        for node_id, node_data in data['nodes'].items():
            node = TaskNode.from_dict(node_data)
            manager.nodes[node_id] = node
        
        return manager
    
    def get_statistics(self) -> Dict[str, Any]:
        """Get tree statistics."""
        status_counts = {}
        for node in self.nodes.values():
            status = node.status.value
            status_counts[status] = status_counts.get(status, 0) + 1
        
        return {
            'total_nodes': len(self.nodes),
            'status_counts': status_counts,
            'leaf_nodes': len(self.get_leaf_nodes()),
            'candidate_tasks': len(self.get_candidate_tasks())
        } 

In [65]:
%pip install colorama

Note: you may need to restart the kernel to use updated packages.


In [66]:
"""PTT Reasoning Module for LLM-based task tree management."""

import json
import re
from typing import Dict, List, Optional, Any, Tuple
from colorama import Fore, Style


class PTTReasoningModule:
    """Handles LLM interactions for PTT management and decision making."""
    
    def __init__(self, tree_manager: TaskTreeManager):
        """
        Initialize the PTT reasoning module.
        
        Args:
            tree_manager: The task tree manager instance
        """
        self.tree_manager = tree_manager
    
    def get_tree_initialization_prompt(self, goal: str, target: str, constraints: Dict[str, Any], available_tools: List[str] = None) -> str:
        """
        Generate prompt for tree initialization.
        
        Args:
            goal: Primary penetration testing objective
            target: Target system/network
            constraints: Scope constraints and limitations
            available_tools: List of available MCP tools
            
        Returns:
            Tree initialization prompt
        """
        tool_info = ""
        if available_tools:
            tool_info = f"""
Available MCP Tools: {', '.join(available_tools)}

You must work within the capabilities of these connected tools. Think about what each tool can accomplish:
- Consider the full capabilities of each available tool
- Adapt your approach to work with available tools
- Be creative in how you leverage available tools
"""
        else:
            tool_info = """
No MCP tools are currently connected. Design an approach that describes the security testing objectives without tool dependencies.
"""

        prompt = f"""You are an autonomous security agent initializing a Pentesting Task Tree (PTT) for a security assessment.

ASSESSMENT CONTEXT:
Goal: {goal}
Target: {target}
Constraints: {json.dumps(constraints, indent=2)}

{tool_info}

TASK:
Analyze this goal and determine what structure and initial tasks are needed to accomplish it efficiently. 

DO NOT assume any predefined phases or structure. Instead:
1. Analyze what the goal actually requires
2. Determine if you need phases/categories or if direct tasks are better
3. Create an appropriate initial structure
4. Define specific actionable tasks to start with

Consider:
- What does this specific goal require?
- What's the minimal viable approach?
- How can available tools be leveraged?
- What structure makes sense for THIS goal?

IMPORTANT: When suggesting tool usage, be specific about commands and modules. For example:

Provide your analysis and initial structure in JSON format:

{{
    "analysis": "Your assessment of what this goal requires and approach",
    "structure": [
        {{
            "type": "phase/category/direct",
            "name": "Name of organizational structure",
            "description": "What this encompasses",
            "justification": "Why this structure element is needed for this goal"
        }}
    ],
    "initial_tasks": [
        {{
            "description": "Specific actionable task",
            "parent": "Which structure element this belongs to, or 'root' for direct tasks",
            "tool_suggestion": "Which available tool to use, or 'manual' if no suitable tool",
            "priority": 1-10,
            "risk_level": "low/medium/high",
            "rationale": "Why this task is necessary for the goal"
        }}
    ]
}}

BE INTELLIGENT: If the goal is simple, don't create complex multi-phase structures. If it's complex, then structure appropriately. Let the goal drive the structure, not the other way around."""

        return prompt
    
    def get_tree_update_prompt(self, tool_output: str, command: str, node: TaskNode) -> str:
        """
        Generate prompt for updating the tree based on tool output.
        
        Args:
            tool_output: Output from the executed tool
            command: The command that was executed
            node: The node being updated
            
        Returns:
            Update prompt
        """
        current_tree = self.tree_manager.to_natural_language()
        
        prompt = f"""You are managing a Pentesting Task Tree (PTT). A task has been executed and you need to update the tree based on the results.

Current PTT State:
{current_tree}

Executed Task: {node.description}
Command: {command}
Tool Output:
{tool_output[:2000]}  # Limit output length

Based on this output, provide updates in the following JSON format:

{{
    "node_updates": {{
        "status": "completed/failed/vulnerable/not_vulnerable",
        "findings": "Summary of key findings from the output",
        "output_summary": "Brief technical summary"
    }},
    "new_tasks": [
        {{
            "description": "New task based on findings",
            "parent_phase": "Phase 1/2/3/4",
            "tool_suggestion": "Suggested tool",
            "priority": 1-10,
            "risk_level": "low/medium/high",
            "rationale": "Why this task is important"
        }}
    ],
    "insights": "Any strategic insights or patterns noticed"
}}

Consider:
1. What vulnerabilities or opportunities were discovered?
2. What follow-up actions are needed based on the findings?
3. Should any new attack vectors be explored?
4. Are there any security misconfigurations evident?"""

        return prompt
    
    def get_next_action_prompt(self, available_tools: List[str]) -> str:
        """
        Generate prompt for selecting the next action.
        
        Args:
            available_tools: List of available MCP tools
            
        Returns:
            Next action selection prompt
        """
        current_tree = self.tree_manager.to_natural_language()
        candidates = self.tree_manager.get_candidate_tasks()
        
        # Prepare candidate descriptions
        candidate_desc = []
        for i, task in enumerate(candidates[:10]):  # Limit to top 10
            desc = f"{i+1}. {task.description}"
            if task.priority:
                desc += f" (Priority: {task.priority})"
            candidate_desc.append(desc)
        
        # Generate tool context
        if available_tools:
            tool_context = f"""
Connected MCP Tools: {', '.join(available_tools)}

Think about how to leverage these tools for the selected task. Each tool has its own capabilities - 
be creative and intelligent about how to accomplish penetration testing objectives with available tools.
If a tool doesn't directly support a traditional approach, consider alternative methods that achieve the same goal.
"""
        else:
            tool_context = """
No MCP tools are currently connected. Select tasks that can be performed manually or recommend connecting appropriate tools.
"""

        prompt = f"""You are managing a Pentesting Task Tree (PTT) and need to select the next action.

Goal: {self.tree_manager.goal}
Target: {self.tree_manager.target}

Current PTT State:
{current_tree}

{tool_context}

Candidate Tasks:
{chr(10).join(candidate_desc)}

Statistics:
- Total tasks: {len(self.tree_manager.nodes)}
- Completed: {sum(1 for n in self.tree_manager.nodes.values() if n.status == NodeStatus.COMPLETED)}
- In Progress: {sum(1 for n in self.tree_manager.nodes.values() if n.status == NodeStatus.IN_PROGRESS)}
- Pending: {sum(1 for n in self.tree_manager.nodes.values() if n.status == NodeStatus.PENDING)}

Select the most strategic next action and provide your response in JSON format:

{{
    "selected_task_index": 1-based index from candidate list,
    "rationale": "Why this task is the best next step",
    "command": "Intelligent request that leverages available tools effectively",
    "tool": "Which available tool to use, or 'manual' if no suitable tool",
    "expected_outcome": "What we hope to discover/achieve",
    "alternative_if_blocked": "Backup task index if this fails"
}}

Consider:
1. Logical progression through the penetration testing methodology
2. Task dependencies and prerequisites
3. Risk vs reward of different approaches
4. How to best utilize available tools for maximum effectiveness
5. Strategic value of each potential action

Be intelligent about tool selection - think about what each available tool can accomplish."""

        return prompt
    
    def get_goal_check_prompt(self) -> str:
        """
        Generate prompt to check if the goal has been achieved.
        
        Returns:
            Goal achievement check prompt
        """
        current_tree = self.tree_manager.to_natural_language()
        goal = self.tree_manager.goal
        
        # Extract completed tasks and findings for better context
        completed_tasks_with_findings = []
        for node in self.tree_manager.nodes.values():
            if node.status == NodeStatus.COMPLETED and node.findings:
                completed_tasks_with_findings.append(f"✓ {node.description}: {node.findings}")
        
        completed_context = "\n".join(completed_tasks_with_findings) if completed_tasks_with_findings else "No completed tasks with findings yet."
        
        prompt = f"""Analyze the current Pentesting Task Tree (PTT) to determine if the PRIMARY GOAL has been achieved.

IMPORTANT: Focus ONLY on whether the specific goal stated has been accomplished. Do not suggest additional scope or activities beyond the original goal.

PRIMARY GOAL: {goal}
Target: {self.tree_manager.target}

COMPLETED TASKS WITH FINDINGS:
{completed_context}

Current PTT State:
{current_tree}

GOAL ACHIEVEMENT CRITERIA:
- For information gathering goals, the goal is achieved when that specific information is obtained
- For vulnerability assessment goals, the goal is achieved when vulnerabilities are identified and documented
- For exploitation goals, the goal is achieved when successful exploitation is demonstrated
- For access goals, the goal is achieved when the specified access level is obtained

Provide your analysis in JSON format:

{{
    "goal_achieved": true/false,
    "confidence": 0-100,
    "evidence": "Specific evidence that the PRIMARY GOAL has been met (quote actual findings)",
    "remaining_objectives": "What still needs to be done if goal not achieved (related to the ORIGINAL goal only)",
    "recommendations": "Next steps ONLY if they relate to the original goal - do not expand scope",
    "scope_warning": "Flag if any tasks seem to exceed the original goal scope"
}}

Consider:
1. Has the SPECIFIC goal been demonstrably achieved?
2. Is there sufficient evidence/proof in the completed tasks?
3. Are there critical paths unexplored that are NECESSARY for the original goal?
4. Would additional testing strengthen the results for the ORIGINAL goal only?

DO NOT recommend expanding the scope beyond the original goal. If the goal is completed, mark it as achieved regardless of what other security activities could be performed."""

        return prompt
    
    def parse_tree_initialization_response(self, llm_response: str) -> Dict[str, Any]:
        """Parse LLM response for tree initialization."""
        try:
            print(f"{Fore.CYAN}Parsing initialization response...{Style.RESET_ALL}")
            # Extract JSON from response
            response_json = self._extract_json(llm_response)
            
            analysis = response_json.get('analysis', 'No analysis provided')
            structure = response_json.get('structure', [])
            initial_tasks = response_json.get('initial_tasks', [])
            
            print(f"{Fore.GREEN}LLM Analysis: {analysis}{Style.RESET_ALL}")
            print(f"{Fore.GREEN}Successfully parsed {len(structure)} structure elements and {len(initial_tasks)} tasks{Style.RESET_ALL}")
            
            return {
                'analysis': analysis,
                'structure': structure,
                'initial_tasks': initial_tasks
            }
        except Exception as e:
            print(f"{Fore.YELLOW}Failed to parse initialization response: {e}{Style.RESET_ALL}")
            print(f"{Fore.YELLOW}Response text (first 500 chars): {llm_response[:500]}{Style.RESET_ALL}")
            return {
                'analysis': 'Failed to parse LLM response',
                'structure': [],
                'initial_tasks': []
            }
    
    def parse_tree_update_response(self, llm_response: str) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
        """Parse LLM response for tree updates."""
        try:
            response_json = self._extract_json(llm_response)
            node_updates = response_json.get('node_updates', {})
            new_tasks = response_json.get('new_tasks', [])
            return node_updates, new_tasks
        except Exception as e:
            print(f"{Fore.YELLOW}Failed to parse update response: {e}{Style.RESET_ALL}")
            return {}, []
    
    def parse_next_action_response(self, llm_response: str, available_tools: List[str] = None) -> Optional[Dict[str, Any]]:
        """Parse LLM response for next action selection."""
        try:
            response_json = self._extract_json(llm_response)
            return response_json
        except Exception as e:
            print(f"{Fore.YELLOW}Failed to parse next action response: {e}{Style.RESET_ALL}")
            return None
    
    def parse_goal_check_response(self, llm_response: str) -> Dict[str, Any]:
        """Parse LLM response for goal achievement check."""
        try:
            response_json = self._extract_json(llm_response)
            return response_json
        except Exception as e:
            print(f"{Fore.YELLOW}Failed to parse goal check response: {e}{Style.RESET_ALL}")
            return {"goal_achieved": False, "confidence": 0}
    
    def _extract_json(self, text: str) -> Dict[str, Any]:
        """Extract JSON from LLM response text."""
        if not text:
            raise ValueError("Empty response text")
        
        print(f"{Fore.CYAN}Attempting to extract JSON from {len(text)} character response{Style.RESET_ALL}")
        
        # Try multiple strategies to extract JSON
        strategies = [
            self._extract_json_code_block,
            self._extract_json_braces,
            self._extract_json_fuzzy,
            self._create_fallback_json
        ]
        
        for i, strategy in enumerate(strategies):
            try:
                result = strategy(text)
                if result:
                    print(f"{Fore.GREEN}Successfully extracted JSON using strategy {i+1}{Style.RESET_ALL}")
                    return result
            except Exception as e:
                print(f"{Fore.YELLOW}Strategy {i+1} failed: {e}{Style.RESET_ALL}")
                continue
        
        raise ValueError("Could not extract valid JSON from response")
    
    def _extract_json_code_block(self, text: str) -> Dict[str, Any]:
        """Extract JSON from code blocks."""
        # Look for JSON between ```json and ``` or just ```
        patterns = [
            r'```json\s*(\{.*?\})\s*```',
            r'```\s*(\{.*?\})\s*```'
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text, re.DOTALL)
            if match:
                json_str = match.group(1)
                return json.loads(json_str)
        
        raise ValueError("No JSON code block found")
    
    def _extract_json_braces(self, text: str) -> Dict[str, Any]:
        """Extract JSON by finding brace boundaries."""
        # Find the first { and last }
        json_start = text.find('{')
        json_end = text.rfind('}')
        
        if json_start != -1 and json_end != -1 and json_end > json_start:
            json_str = text[json_start:json_end + 1]
            return json.loads(json_str)
        
        raise ValueError("No valid JSON braces found")
    
    def _extract_json_fuzzy(self, text: str) -> Dict[str, Any]:
        """Try to extract JSON with more flexible matching."""
        # Look for task-like patterns and try to construct JSON
        if "tasks" in text.lower():
            # Try to find task descriptions
            task_patterns = [
                r'"description":\s*"([^"]+)"',
                r'"tool_suggestion":\s*"([^"]+)"',
                r'"priority":\s*(\d+)',
                r'"risk_level":\s*"([^"]+)"'
            ]
            
            # This is a simplified approach - could be enhanced
            # For now, fall through to the next strategy
            pass
        
        raise ValueError("Fuzzy JSON extraction failed")
    
    def _create_fallback_json(self, text: str) -> Dict[str, Any]:
        """Create fallback JSON if no valid JSON is found."""
        print(f"{Fore.YELLOW}Creating fallback JSON structure{Style.RESET_ALL}")
        
        # Return an empty but valid structure
        return {
            "tasks": [],
            "node_updates": {"status": "completed"},
            "new_tasks": [],
            "selected_task_index": 1,
            "goal_achieved": False,
            "confidence": 0
        }
    
    def verify_tree_update(self, old_tree_state: str, new_tree_state: str) -> bool:
        """
        Verify that tree updates maintain integrity.
        
        Args:
            old_tree_state: Tree state before update
            new_tree_state: Tree state after update
            
        Returns:
            True if update is valid
        """
        # For now, basic verification - can be enhanced
        # Check that only leaf nodes were modified (as per PentestGPT approach)
        # This is simplified - in practice would need more sophisticated checks
        
        return True  # Placeholder - implement actual verification logic
    
    def generate_strategic_summary(self) -> str:
        """Generate a strategic summary of the current PTT state."""
        stats = self.tree_manager.get_statistics()
        
        summary = f"""
=== PTT Strategic Summary ===
Goal: {self.tree_manager.goal}
Target: {self.tree_manager.target}

Progress Overview:
- Total Tasks: {stats['total_nodes']}
- Completed: {stats['status_counts'].get('completed', 0)}
- In Progress: {stats['status_counts'].get('in_progress', 0)}
- Failed: {stats['status_counts'].get('failed', 0)}
- Vulnerabilities Found: {stats['status_counts'].get('vulnerable', 0)}

Current Phase Focus:
"""
        
        # Identify which phase is most active
        phase_activity = {}
        for node in self.tree_manager.nodes.values():
            if node.node_type == "phase":
                completed_children = sum(
                    1 for child_id in node.children_ids
                    if child_id in self.tree_manager.nodes 
                    and self.tree_manager.nodes[child_id].status == NodeStatus.COMPLETED
                )
                total_children = len(node.children_ids)
                phase_activity[node.description] = (completed_children, total_children)
        
        for phase, (completed, total) in phase_activity.items():
            if total > 0:
                progress = (completed / total) * 100
                summary += f"- {phase}: {completed}/{total} tasks ({progress:.0f}%)\n"
        
        # Add key findings
        summary += "\nKey Findings:\n"
        vuln_count = 0
        for node in self.tree_manager.nodes.values():
            if node.status == NodeStatus.VULNERABLE and node.findings:
                vuln_count += 1
                summary += f"- {node.description}: {node.findings[:100]}...\n"
                if vuln_count >= 5:  # Limit to top 5
                    break
        
        return summary
    
    def validate_and_fix_tool_suggestions(self, tasks: List[Dict[str, Any]], available_tools: List[str]) -> List[Dict[str, Any]]:
        """Let the LLM re-evaluate tool suggestions if they don't match available tools."""
        if not available_tools:
            return tasks
        
        # Check if any tasks use unavailable tools
        needs_fixing = []
        valid_tasks = []
        
        for task in tasks:
            tool_suggestion = task.get('tool_suggestion', '')
            if tool_suggestion in available_tools or tool_suggestion in ['manual', 'generic']:
                valid_tasks.append(task)
            else:
                needs_fixing.append(task)
        
        if needs_fixing:
            print(f"{Fore.YELLOW}Some tasks reference unavailable tools. Letting AI re-evaluate...{Style.RESET_ALL}")
            # Return all tasks - let the execution phase handle tool mismatches intelligently
            
        return tasks 

In [67]:
def pretty_print_result(result):
    print("\n" + "="*60)
    print("FINAL AGENT OUTPUT")
    print("="*60)

    for msg in result["messages"]:
        if msg.__class__.__name__ == "HumanMessage":
            print("\n🧑 USER:")
            print(msg.content)

        elif msg.__class__.__name__ == "AIMessage":
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                print("\n🤖 ASSISTANT (Tool Call Request):")
                print(f"  → Tool: {msg.tool_calls[0]['name']}")
                print(f"  → Args: {msg.tool_calls[0]['args']}")
            else:
                print("\n🤖 ASSISTANT:")
                print(msg.content)

        elif msg.__class__.__name__ == "ToolMessage":
            print("\n🛠️ TOOL RESPONSE:")
            print(msg.content)

        else:
            print("\n❓ UNKNOWN MESSAGE TYPE:")
            print(msg)
    
    print("\n" + "="*60 + "\n")



In [68]:

# mcp_server = load_config()
# stack, GLOBAL_SCHEMA, tools, GLOBAL_NAME_TO_TOOL = await configure_mcp(mcp_server)



# llm = ChatOpenAI( 
#     model="gpt-4o",
#       openai_api_key=os.getenv("OPEN_AI_API_KEY"), 
#       openai_api_base=os.getenv("OPEN_AI_API_BASE"),
#     ).bind_tools(tools)


# class AgentState(TypedDict):
#     messages: Annotated[list[BaseMessage], add_messages]
#     tool_calls = []


# def model_call(state: AgentState):

#     sys_prompt = SystemMessage(content="""
# You are a precise cybersecurity agent with acess to differnet tools when executing any tool if you get wrong schema correct it
# with the validation meassage and try again and always summarize the final output after the tool calls.
# """)

#     llm_input = [sys_prompt] + state["messages"]
#     response: AIMessage = llm.invoke(llm_input)


#     print(f"Reasoner: {response}")

#     return {
#         "messages": [response],
#     }



# def run_tool(state: AgentState):
#     last = state["messages"][-1]

#     tool_messages = []

#     for tc in last.tool_calls:
#         tool_id = tc["id"]
#         tool_name = tc["name"]
#         tool_args = tc["args"]

#         validation = validate_arguments(tool_args, GLOBAL_SCHEMA[tool_name])
#         if validation != "Valid":
#             print(validation)
#             tool_messages.append(
#                 ToolMessage(
#                     content=validation,
#                     name=tool_name,
#                     tool_call_id=tool_id
#                 )
#             )
#         else:
#             result = GLOBAL_NAME_TO_TOOL[tool_name].run(tool_args)
#             tool_messages.append(
#                 ToolMessage(
#                     content=result.content[0].text,
#                     name=tool_name,
#                     tool_call_id=tool_id
#                 )
#             )


#     return {
#         "messages": tool_messages,
#     }



# def route_reasoner(state: AgentState):
#     last = state["messages"][-1]
#     if isinstance(last, AIMessage) and last.tool_calls:
#         return "tools"
#     return END


# graph = StateGraph(AgentState)

# graph.add_node("reasoner", model_call)
# graph.add_node("run_tool", run_tool)

# graph.add_edge(START, "reasoner")


# graph.add_conditional_edges(
#     "reasoner",
#     route_reasoner,
#     {"tools": "run_tool", END: END},
# )


# graph.add_edge("run_tool", "reasoner")

# react = graph.compile(debug=False)


# conversation_history = []
# user_input = input("Enter: ")
# while user_input.lower() != "exit":
#     conversation_history.append(HumanMessage(content=user_input))
    
#     # Initialize tools_run on first invoke
#     state_input = {
#         "messages": conversation_history,
#     }
    
#     result = react.invoke(state_input)
    
#     print(f"User: {user_input}")
#     print(f"AI: {result['messages'][-1].content}")
    

#     user_input = input("Enter: ")

# await stack.aclose()



In [69]:
import time
from openai import RateLimitError

In [70]:
import os
from typing import List


# MCP (Modular Cybersecurity Platform) setup
mcp_server = load_config()
stack, GLOBAL_SCHEMA, tools, GLOBAL_NAME_TO_TOOL = await configure_mcp(mcp_server)

# Initialize LLM
llm = ChatOllama(
    model="llama3.2",
    temperature=0
).bind_tools(tools)

# === Initialize Tree and Reasoning Module ===
goal = "Assess the security posture of example.com"
target = "https://example.com"
constraints = {"scope": "public endpoints only"}

tree_manager = TaskTreeManager()
tree_manager.initialize_tree(goal, target, constraints)
reasoning_module = PTTReasoningModule(tree_manager)

available_tools = list(GLOBAL_NAME_TO_TOOL.keys())

# === Get Initial Tasks from LLM ===
init_prompt = reasoning_module.get_tree_initialization_prompt(goal, target, constraints, available_tools)
init_msg = HumanMessage(content=init_prompt)

init_response = llm.invoke([SystemMessage(content="You are a cybersecurity agent."), init_msg])
parsed_init = reasoning_module.parse_tree_initialization_response(init_response.content)

# Add initial tasks to tree
for task in parsed_init["initial_tasks"]:
    node = TaskNode(
        description=task["description"],
        parent_id=tree_manager.root_id,
        priority=task.get("priority", 5),
        risk_level=task.get("risk_level", "low")
    )
    tree_manager.add_node(node)

# === Task Execution Loop ===
while True:
    # 1. Get candidate tasks
    candidates = tree_manager.get_candidate_tasks()
    if not candidates:
        print("All tasks completed or blocked.")
        break

    # 2. Ask LLM to choose next task
    next_action_prompt = reasoning_module.get_next_action_prompt(available_tools)
    for _ in range(5):
        try:
            next_response = llm.invoke([SystemMessage(content="Select next task"), HumanMessage(content=next_action_prompt)])
            break
        except RateLimitError:
            print("Rate limit hit. Waiting 5 seconds...")
            time.sleep(5)
    next_action = reasoning_module.parse_next_action_response(next_response.content)

    selected_index = next_action.get("selected_task_index", 1) - 1
    selected_task = candidates[selected_index]
    print(f"\n=== Executing Task: {selected_task.description} ===")

    # 3. Execute tool if suggested
    tool_name = next_action.get("tool", "manual")
    command = next_action.get("command", "")
    if tool_name in GLOBAL_NAME_TO_TOOL:
        tool_args = {"command": command}
        
        # Validate arguments first
        validation = validate_arguments(tool_args, GLOBAL_SCHEMA[tool_name])
        if validation != "Valid":
            print(f"⚠ Argument validation failed: {validation}")
            tool_output = f"Tool execution skipped due to invalid arguments: {validation}"
        else:
            # Safe execution
            try:
                result = GLOBAL_NAME_TO_TOOL[tool_name].run(tool_args)
                tool_output = result.content[0].text
            except Exception as e:
                print(f"❌ Tool execution error: {e}")
                tool_output = f"Tool execution failed with error: {e}"
    else:
        tool_output = "Manual task, no tool executed."



    # 4. Update tree using LLM
    update_prompt = reasoning_module.get_tree_update_prompt(tool_output, command, selected_task)
    for _ in range(5):
        try:
            update_response = llm.invoke([SystemMessage(content="Update tree based on output"), HumanMessage(content=update_prompt)])
            break
        except RateLimitError:
            print("Rate limit hit. Waiting 5 seconds...")
            time.sleep(5)
    
    node_updates, new_tasks = reasoning_module.parse_tree_update_response(update_response.content)

    # Apply updates
    tree_manager.update_node(selected_task.id, node_updates)

    # Add new tasks if any
    for t in new_tasks:
        node = TaskNode(
            description=t["description"],
            parent_id=tree_manager.root_id,
            priority=t.get("priority", 5),
            risk_level=t.get("risk_level", "low")
        )
        tree_manager.add_node(node)

    # 5. Print strategic summary
    print(reasoning_module.generate_strategic_summary())

    # 6. Check if goal achieved
    goal_check_prompt = reasoning_module.get_goal_check_prompt()
    for _ in range(5):
        try:
            goal_response = llm.invoke([SystemMessage(content="Check goal achievement"), HumanMessage(content=goal_check_prompt)])
            break
        except RateLimitError:
            print("Rate limit hit. Waiting 5 seconds...")
            time.sleep(5)
    goal_status = reasoning_module.parse_goal_check_response(goal_response.content)
    if goal_status.get("goal_achieved", False):
        print("\n🎯 Goal Achieved!")
        break

# Close MCP stack
await stack.aclose()


Connecting to server nmap...
Session initialized for nmap
Connecting to server sqlmap...
Session initialized for sqlmap
Connecting to server ffuf...
Session initialized for ffuf
Connecting to server masscan...
Session initialized for masscan
Connecting to server sslscan...
Session initialized for sslscan
Schema Dumped at Input_schema.json
Parsing initialization response...
Attempting to extract JSON from 2802 character response
Strategy 1 failed: No JSON code block found
Successfully extracted JSON using strategy 2
LLM Analysis: The goal requires assessing the security posture of example.com by analyzing public endpoints only. This involves identifying open ports, detecting potential vulnerabilities, and evaluating SSL/TLS configurations. The available tools can be leveraged to achieve this goal efficiently.
Successfully parsed 3 structure elements and 4 tasks
Failed to parse next action response: Empty response text


AttributeError: 'NoneType' object has no attribute 'get'